# 05 - Aplicaciones Prácticas: Comparación Teammates, Puesta a Punto y Estrategia
**Tesis de Maestría en Ciencia de Datos - Universidad Austral**  
*Autor: Nicolás Palavecino | Tutor: Ezequiel Nuske*

---

## 1. Aplicaciones Prácticas según el Plan de Trabajo Final
En este notebook se abordan los tres objetivos específicos de la tesis:
1. **Comparación Intra-Equipo (*Teammates*)**: Comparar pilotos del mismo equipo para aislar el factor humano y diseñar puestas a punto (*setups*) personalizadas.
2. **Optimización de Inversiones y Desarrollo Técnico**: Identificar la sinergia entre la filosofía técnica del monoplaza y el estilo del piloto.
3. **Scouting y Compatibilidad de Contratación**: Marco objetivo para emparejar perfiles de conducción de pilotos potenciales con la ventana operativa del coche.
4. **Sinergia con Degradación de Neumáticos**: Relación entre agresividad de frenado/tracción y pendiente de degradación en carrera.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from f1_style_clustering.data.loader import load_session, enable_cache
from f1_style_clustering.features.zoi import extract_circuit_zones
from f1_style_clustering.features.aggregators import build_session_feature_matrix
from f1_style_clustering.features.preprocessor import normalize_relative_to_mean, standardize_features
from f1_style_clustering.models.clustering import fit_kmeans
from f1_style_clustering.analysis.archetypes import characterize_clusters, assign_archetype_names
from f1_style_clustering.analysis.teammates import compare_teammates, compute_intra_team_deltas
from f1_style_clustering.visualization.radar_charts import plot_driver_radar_comparison

enable_cache('cache')
session = load_session(year=2025, grand_prix='Italian Grand Prix', session_type='R')
analysis_zones = extract_circuit_zones(session)
df_raw = build_session_feature_matrix(session, analysis_zones=analysis_zones)
df_norm, df_meta = normalize_relative_to_mean(df_raw)
df_standardized, _, _ = standardize_features(df_norm)
cluster_labels, _ = fit_kmeans(df_standardized, n_clusters=3)


## 2. Comparación Intra-Equipo (*Teammates* en Monoplazas Idénticos)

In [ ]:
# Calcular distancias euclidianas y deltas de estilo entre compañeros de equipo
teammates_df = compare_teammates(df_standardized, df_meta)
print('--- Distancias de Estilo Intra-Equipo ---')
display(teammates_df[['Team', 'Driver_1', 'Driver_2', 'Style_Euclidean_Distance']].sort_values(by='Style_Euclidean_Distance', ascending=False))


## 3. Análisis de Detalle de Pareja de Compañeros (ej. McLaren / Ferrari / Mercedes)

In [ ]:
# Seleccionar una pareja de compañeros para análisis detallado
if not teammates_df.empty:
    sample_pair = teammates_df.iloc[0]
    d1, d2 = sample_pair['Driver_1'], sample_pair['Driver_2']
    print(f"Analizando deltas detallados: {d1} vs {d2} ({sample_pair['Team']})")
    
    df_deltas = compute_intra_team_deltas(df_standardized, d1, d2)
    display(df_deltas.head(8))
    
    # Gráfico de radar comparativo
    radar_feats = [c for c in df_standardized.columns if 'ApexSpeed' in c or 'MaxDecel' in c or 'ThrottleRate' in c][:6]
    fig_comp = plot_driver_radar_comparison(df_standardized, drivers=[d1, d2], features=radar_feats, title=f"Comparación de Estilo: {d1} vs {d2}")
    plt.show()


## 4. Matriz de Compatibilidad para Scouting y Scouting de Talento

In [ ]:
# Matriz de distancia euclidiana entre todos los pilotos (para encontrar perfiles análogos)
from scipy.spatial.distance import cdist

dist_matrix = pd.DataFrame(
    cdist(df_standardized.values, df_standardized.values, metric='euclidean'),
    index=df_standardized.index,
    columns=df_standardized.index,
)

plt.figure(figsize=(11, 9))
sns.heatmap(dist_matrix, cmap='viridis_r', annot=True, fmt='.1f')
plt.title('Matriz de Disimilitud Euclidiana de Estilo de Conducción entre Pilotos', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
